In [13]:
# =============================================================================
# CARGAR SECRETS
# =============================================================================
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
drive_id = user_secrets.get_secret("DRIVE_SHOP_IND_KERAS")
pat = user_secrets.get_secret("GITHUB_PAT")


In [ ]:
# =============================================================================
# CONEXION CON DRIVE Y RECUPERAMOS SESION DE KERAS TUNER
# =============================================================================

# ------- PARA GOOGLE COLAB ---------------
# from google.colab import drive
# path = drive.mount('/content/drive')
# print('Path to competition files:', path)

# # Crea el directorio local si no existe
# !rm -rf kt_dir
# !mkdir -p kt_dir

# # Copia el contenido desde Drive al directorio local de Colab
# # Descomprimir el zip directamente desde Drive al entorno local
# !unzip -q "/content/drive/MyDrive/Colab Notebooks/hypermodel_sessions/lab_04/kt_dir.zip" -d ./

# ------- PARA KAGGLE ---------------
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# drive_id = user_secrets.get_secret("DRIVE_KTDIR_ZIP")

# !pip install gdown
# !gdown --id {drive_id}
# !unzip -q "/kaggle/working/kt_dir.zip" -d ./

In [14]:
# =============================================================================
# CONEXION CON REPOSITORIO PARA ACCEDER A FUNCIONES Y MODULOS DEL SRC
# =============================================================================
# ------ PARA GOOGLE COLAB ----------
# from google.colab import userdata

# # Obtener el token de forma segura
# GITHUB_PAT = userdata.get('GithubPat')

# # Usar f-string para construir el comando git clone con el token
# !git clone https://{GITHUB_PAT}@github.com/Franku03/ML-jupyter-notebooks.git


# ------ PARA KAGGLE ----------
# Obtener el token y otros datos necesarios
pat = user_secrets.get_secret("GITHUB_PAT")

!git clone https://{pat}@github.com/Franku03/ml-shopping-indicator.git

fatal: destination path 'ml-shopping-indicator' already exists and is not an empty directory.


In [15]:
# =============================================================================
# CARGA DE LIBRERIAS
# =============================================================================

!pip install polars pyarrow keras --upgrade
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# =============================================================================
# CARGA DE DATASETS
# =============================================================================

# LOCAL
# shop_df = pd.read_csv("../datasets/train_short.csv")

# TODO: Cambiar a polars
# KAGGLE
shop_df = pd.read_csv(
    "/kaggle/working/ml-shopping-indicator/datasets/dataset shop.csv", low_memory=False
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 17.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 33.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: polars-runtime-32
    Found existing installation: polars-runtime-32 1.35.2
    Uninstalling polars-runtime-32-1.35.2:
      Successfully uninstalled polars-runtime-32-1.35.2
  Attempting uninstall: polars
    Found existing installation: polars 1.35.2
    Uninstalling polars-1.35.2:
      Successfully uninstalled polars-1.35.2
  Attempting uninstall: keras
    Found existing installation: k

In [16]:
shop_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           